In [1]:
%pip install xgboost

Looking in indexes: http://192.168.0.94:8888/repository/pypi/simple
     |████████████████████████████████| 255.9 MB 101.4 MB/s eta 0:00:01     |████████████████████████████▎   | 226.6 MB 101.4 MB/s eta 0:00:01
You should consider upgrading via the '/home/ma-user/anaconda3/envs/PyTorch-1.8/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [2]:

import pandas as pd
import joblib
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report


/home/ma-user/anaconda3/envs/PyTorch-1.8/lib/python3.7/site-packages/pkg_resources/__init__.py:126: PkgResourcesDeprecationWarning: 2.1.0.5d9c87c8 is an invalid version and will not be supported in a future release
  PkgResourcesDeprecationWarning,


In [3]:
import moxing as mox
mox.file.copy('obs://datasets-cffe/students_churn.csv', 'students_churn.csv')

INFO:root:Using MoXing-v2.1.0.5d9c87c8-5d9c87c8
INFO:root:Using OBS-Python-SDK-3.20.9.1
/home/ma-user/anaconda3/envs/PyTorch-1.8/lib/python3.7/site-packages/requests/__init__.py:104: RequestsDependencyWarning: urllib3 (1.26.12) or chardet (5.2.0)/charset_normalizer (2.0.12) doesn't match a supported version!
  RequestsDependencyWarning)


In [4]:
df = pd.read_csv('students_churn.csv', sep=";")

In [5]:
df.head()

,Marital status,Application mode,Application order,Course,Daytime/evening attendance\t,Previous qualification,Previous qualification (grade),Nacionality,Mother's qualification,Father's qualification,...,Curricular units 2nd sem (credited),Curricular units 2nd sem (enrolled),Curricular units 2nd sem (evaluations),Curricular units 2nd sem (approved),Curricular units 2nd sem (grade),Curricular units 2nd sem (without evaluations),Unemployment rate,Inflation rate,GDP,Target
0,1,17,5,171,1,1,122.0,1,19,12,...,0,0,0,0,0.000000,0,10.8,1.4,1.74,Dropout
1,1,15,1,9254,1,1,160.0,1,1,3,...,0,6,6,6,13.666667,0,13.9,-0.3,0.79,Graduate
2,1,1,5,9070,1,1,122.0,1,37,37,...,0,6,0,0,0.000000,0,10.8,1.4,1.74,Dropout
3,1,17,2,9773,1,1,122.0,1,38,37,...,0,6,10,5,12.400000,0,9.4,-0.8,-3.12,Graduate
4,2,39,1,8014,0,1,100.0,1,37,38,...,0,6,6,6,13.000000,0,13.9,-0.3,0.79,Graduate


# Clean Format

In [6]:
df['Target'] = df['Target'].str.replace('"', '')

In [7]:
df.to_csv("students_churn_formatted.csv", index=False)

In [8]:
import moxing as mox
mox.file.copy( 'students_churn_formatted.csv', 'obs://datasets-cffe/students_churn_formatted.csv')

# Preprocess Data

In [9]:
df['Target'] = df['Target'].apply(lambda x: 0 if x=='Dropout' else 1)

#

In [10]:
numeric_cols = df.select_dtypes(include='number').columns
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())


In [11]:
features = df.drop(columns=['Target'])
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)
scaled_df = pd.DataFrame(scaled_features, columns=features.columns)
scaled_df['Target'] = df['Target']


In [12]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4424 entries, 0 to 4423
Data columns (total 37 columns):
 #   Column                                          Non-Null Count  Dtype  
---  ------                                          --------------  -----  
 0   Marital status                                  4424 non-null   int64  
 1   Application mode                                4424 non-null   int64  
 2   Application order                               4424 non-null   int64  
 3   Course                                          4424 non-null   int64  
 4   Daytime/evening attendance	                     4424 non-null   int64  
 5   Previous qualification                          4424 non-null   int64  
 6   Previous qualification (grade)                  4424 non-null   float64
 7   Nacionality                                     4424 non-null   int64  
 8   Mother's qualification                          4424 non-null   int64  
 9   Father's qualification                   

# Feature Engineering

In [13]:
df['Avg_Grades'] = (df['Curricular units 1st sem (grade)'] + df['Curricular units 2nd sem (grade)']) / 2
df['Approval_Rate'] = (
    (df['Curricular units 1st sem (approved)'] + df['Curricular units 2nd sem (approved)']) /
    (df['Curricular units 1st sem (enrolled)'] + df['Curricular units 2nd sem (enrolled)']).replace(0, 1)
)

# Visualize Features

# Train and Test Split

In [14]:
import pandas as pd
from sklearn.model_selection import train_test_split
import os

In [15]:
X = df.drop("Target", axis=1)
y = df["Target"]


In [16]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Train

In [17]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [18]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=200, random_state=42),
    "XGBoost": XGBClassifier(use_label_encoder=False, eval_metric='logloss', random_state=42)
}

In [19]:

results = {}

for name, model in models.items():
    print(f"\n🚀 Training {name}...")
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results[name] = acc
    print(f"✅ {name} Accuracy: {acc:.4f}")


🚀 Training Logistic Regression...
✅ Logistic Regression Accuracy: 0.8780

🚀 Training Random Forest...
✅ Random Forest Accuracy: 0.8712

🚀 Training XGBoost...
✅ XGBoost Accuracy: 0.8667


In [20]:
best_model_name = max(results, key=results.get)
best_model = models[best_model_name]
best_acc = results[best_model_name]

print(f"\n🏆 Best Model: {best_model_name} (Accuracy: {best_acc:.4f})")


🏆 Best Model: Logistic Regression (Accuracy: 0.8780)


In [21]:
import os
from pathlib import Path
import torch
import torch.nn as nn
import joblib
import moxing as mox
import numpy as np

# ------------------------------
# 1️⃣ Wrap Logistic Regression in PyTorch
# ------------------------------
class SklearnLogisticWrapper(nn.Module):
    def __init__(self, sklearn_model, scaler):
        super(SklearnLogisticWrapper, self).__init__()
        self.model = sklearn_model
        self.scaler = scaler

    def forward(self, x):
        # Convert torch tensor to numpy
        x_np = x.detach().cpu().numpy()
        # Scale
        x_scaled = self.scaler.transform(x_np)
        # Predict
        y_pred = self.model.predict(x_scaled)
        # Convert to torch tensor
        return torch.tensor(y_pred, dtype=torch.long)

# Wrap your trained model
wrapped_model = SklearnLogisticWrapper(best_model, scaler)

# ------------------------------
# 2️⃣ Create folder structure
# ------------------------------
base_dir = Path("meta_model_v1")
model_dir = base_dir / "model"
inference_dir = base_dir / "inference"
config_dir = base_dir / "config"

for d in [model_dir, inference_dir, config_dir]:
    d.mkdir(parents=True, exist_ok=True)

# ------------------------------
# 3️⃣ Save the wrapped model as .pt and the scaler/model as .pkl in model_dir
# ------------------------------
pt_model_path = model_dir / "best_model.pt"
torch.save(wrapped_model, pt_model_path)
print(f"✅ Wrapped model saved as {pt_model_path}")

# Save scaler and original sklearn model in the same folder
pkl_scaler_path = model_dir / "scaler.pkl"
pkl_model_path = model_dir / "best_model.pkl"

joblib.dump(best_model, pkl_model_path)
joblib.dump(scaler, pkl_scaler_path)

print(f"✅ Scaler saved as {pkl_scaler_path}")
print(f"✅ Sklearn model saved as {pkl_model_path}")

# ------------------------------
# 4️⃣ Create inference.py for PyTorch deployment
# ------------------------------
inference_code = """
import torch
import numpy as np

# Load the wrapped PyTorch model
model = torch.load("model/best_model.pt")
model.eval()

def predict(input_data):
    x = torch.tensor(input_data, dtype=torch.float32).reshape(1, -1)
    with torch.no_grad():
        y_pred = model(x)
    return int(y_pred.item())

if __name__ == "__main__":
    sample_input = [
        1,0,1,3,1,2,15.0,1,2,2,3,4,16.5,0,0,1,1,0,19,0,
        6,6,6,5,14.0,0,6,6,6,5,14.0,0,0.05,0.02,50000,1
    ]
    print("Prediction:", predict(sample_input))
"""

(inference_dir / "inference.py").write_text(inference_code.strip())
print("✅ inference.py created.")

# ------------------------------
# 5️⃣ Create model.yaml for PyTorch engine
# ------------------------------
model_yaml = """
model_name: student_performance_model
version: 0.0.1
ai_engine: PyTorch
runtime_dependencies:
  - torch>=1.8.0
  - numpy
inference:
  entry: inference/inference.py
  api:
    protocol: HTTPS
    port: 8080
    input:
      - name: input_data
        type: list
    output:
      - name: prediction
        type: int
deployment_type: Real-Time
"""

(config_dir / "model.yaml").write_text(model_yaml.strip())
print("✅ model.yaml created.")

# ------------------------------
# 6️⃣ Upload all files to OBS using moxing
# ------------------------------
obs_base = "obs://datasets-cffe/meta_model_v1/"

def upload_to_obs(local_path, obs_path):
    mox.file.copy(str(local_path), obs_path)
    print(f"Uploaded {local_path.name} → {obs_path}")

# Upload all files in model_dir
for file in model_dir.iterdir():
    upload_to_obs(file, obs_base + f"model/{file.name}")

# Upload inference.py
upload_to_obs(inference_dir / "inference.py", obs_base + "inference/inference.py")

# Upload model.yaml
upload_to_obs(config_dir / "model.yaml", obs_base + "config/model.yaml")

print("✅ All files uploaded to OBS successfully!")


✅ Wrapped model saved as meta_model_v1/model/best_model.pt
✅ Scaler saved as meta_model_v1/model/scaler.pkl
✅ Sklearn model saved as meta_model_v1/model/best_model.pkl
✅ inference.py created.
✅ model.yaml created.
Uploaded scaler.pkl → obs://datasets-cffe/meta_model_v1/model/scaler.pkl
Uploaded best_model.pt → obs://datasets-cffe/meta_model_v1/model/best_model.pt
Uploaded best_model.pkl → obs://datasets-cffe/meta_model_v1/model/best_model.pkl
Uploaded inference.py → obs://datasets-cffe/meta_model_v1/inference/inference.py
Uploaded model.yaml → obs://datasets-cffe/meta_model_v1/config/model.yaml
✅ All files uploaded to OBS successfully!


/home/ma-user/anaconda3/envs/PyTorch-1.8/lib/python3.7/site-packages/tqdm/auto.py:22: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [22]:

print("💾 Best model and scaler saved successfully!")

print("\n📋 Classification Report:")
y_pred_best = best_model.predict(X_test_scaled)
print(classification_report(y_test, y_pred_best))


💾 Best model and scaler saved successfully!

📋 Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.73      0.79       284
           1       0.88      0.95      0.91       601

    accuracy                           0.88       885
   macro avg       0.88      0.84      0.85       885
weighted avg       0.88      0.88      0.87       885



In [23]:
model = joblib.load("best_model.pkl")
scaler = joblib.load("scaler.pkl")



In [24]:
y_pred_proba = model.predict_proba(X_test)[:,1]
y_pred_class = model.predict(X_test)

results = X_test.copy()
results['y_true'] = y_test
results['y_pred_proba'] = y_pred_proba
results['y_pred_class'] = y_pred_class

In [25]:

results.to_csv("churn_predictions.csv", index=False)
print("✅ Churn predictions saved.")

✅ Churn predictions saved.
